# Biohub A+B raw proposal export — all 199 training videos

Exports shared A+B detections before edge prediction, ILP, or post-processing. These proposals are inputs for sparse-GT-aware association fine-tuning.


In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
_train_dir_override = os.environ.get("BIOHUB_TRAIN_DIR", "").strip()
TRAIN_DIR = Path(_train_dir_override) if _train_dir_override else COMP_DIR / "train"
# Compatibility for the inherited artifact/metadata checks.
TEST_DIR = TRAIN_DIR

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"
REGISTRATION_DIAGNOSTICS_PATH = WORKING_DIR / "registration_diagnostics.csv"
DENSE_SHORT7_COMPONENTS_PATH = WORKING_DIR / "dense_short7_components.csv"
DENSE_SHORT7_NODES_PATH = WORKING_DIR / "dense_short7_nodes.csv"

METHOD = "unet_transformer"

# --- TRUE PRE-ILP ENSEMBLE (three slots; A+B enabled, C ready) ---
# Each enabled model sees the same images. Detection evidence is fused before
# peak extraction; edge evidence is fused on one shared graph before one ILP.
ENSEMBLE_MODELS = [
    {
        "name": "A",
        "enabled": True,
        "path": "/kaggle/input/datasets/tweakai/biohub-ensemble-2modelv3/weights/unet_transformer/split_0/edge_predictor_best.pth",
        "weight": float(os.environ.get("BIOHUB_MODEL_A_WEIGHT", "1.0")),
        "device": os.environ.get("BIOHUB_MODEL_A_DEVICE", "cuda:0"),
    },
    {
        "name": "B",
        "enabled": True,
        "path": "/kaggle/input/datasets/tweakai/biohub-ensemble-2modelv3/weights/unet_transformer/split_1/edge_predictor_best.pth",
        "weight": float(os.environ.get("BIOHUB_MODEL_B_WEIGHT", "1.0")),
        "device": os.environ.get("BIOHUB_MODEL_B_DEVICE", "cuda:1"),
    },
    {
        "name": "C",
        "enabled": os.environ.get("BIOHUB_MODEL_C_ENABLED", "0") == "1",
        "path": os.environ.get("BIOHUB_MODEL_C_PATH", "/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/weights/unet_transformer/split_0/edge_predictor_best.pth"),
        "weight": float(os.environ.get("BIOHUB_MODEL_C_WEIGHT", "0.5")),
        # Kaggle has two GPUs, so a future C defaults to sharing GPU 0.
        "device": os.environ.get("BIOHUB_MODEL_C_DEVICE", "cuda:0"),
    },
]
WEIGHTS_RELATIVE = ENSEMBLE_MODELS[0]["path"]

EXPERIMENT_TAG = "ab_raw_proposal_export_199_v1"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"
DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))
EDGE_THRESHOLD = float(os.environ.get("BIOHUB_EDGE_THRESHOLD", "0.54"))
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.5"))

# Empty for a real submission. Useful for local smoke tests, e.g. BIOHUB_SLICE=:1.
SLICE = os.environ.get("BIOHUB_SLICE", "").strip()

# If dependencies are not already installed and no offline wheels are attached,
# this controls whether the notebook attempts PyPI installation.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"
RUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"
RUN_VISUAL_EDA = os.environ.get("BIOHUB_RUN_VISUAL_EDA", "0") != "0"
RUN_DENSE_SHORT7_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_DENSE_SHORT7_DIAGNOSTICS", "1") != "0"

# Output-level graph post-processing.
OUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))
MOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "9.5"))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.52"))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.78"))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2800"))

# Confidence-gated global frame registration. Only abrupt, well-supported shifts
# are applied; low-confidence and ordinary-motion frames use the exact .906 path.
OUTPUT_REGISTRATION_AWARE = os.environ.get("BIOHUB_OUTPUT_REGISTRATION_AWARE", "1") != "0"
REGISTRATION_DOWNSAMPLE_Z = int(os.environ.get("BIOHUB_REGISTRATION_DOWNSAMPLE_Z", "1"))
REGISTRATION_DOWNSAMPLE_YX = int(os.environ.get("BIOHUB_REGISTRATION_DOWNSAMPLE_YX", "4"))
REGISTRATION_MIN_PSR = float(os.environ.get("BIOHUB_REGISTRATION_MIN_PSR", "8.0"))
REGISTRATION_MIN_APPLY_UM = float(os.environ.get("BIOHUB_REGISTRATION_MIN_APPLY_UM", "3.0"))
REGISTRATION_MAX_APPLY_UM = float(os.environ.get("BIOHUB_REGISTRATION_MAX_APPLY_UM", "40.0"))
REGISTRATION_LINEFIT_STABILIZE = os.environ.get("BIOHUB_REGISTRATION_LINEFIT_STABILIZE", "1") != "0"
REGISTRATION_GEOMETRY_VALIDATE = os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_VALIDATE", "1") != "0"
REGISTRATION_GEOMETRY_MIN_NODES = int(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_MIN_NODES", "20"))
REGISTRATION_GEOMETRY_DISTANCE_CAP_UM = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_DISTANCE_CAP_UM", "8.0"))
REGISTRATION_GEOMETRY_COVERAGE_UM = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_COVERAGE_UM", "4.0"))
REGISTRATION_GEOMETRY_VETO_MARGIN_UM = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_VETO_MARGIN_UM", "0.35"))
REGISTRATION_GEOMETRY_VETO_COVERAGE_DROP = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_VETO_COVERAGE_DROP", "0.03"))
REGISTRATION_GEOMETRY_RESCUE_MIN_UM = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_RESCUE_MIN_UM", "2.0"))
REGISTRATION_GEOMETRY_RESCUE_MARGIN_UM = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_RESCUE_MARGIN_UM", "0.25"))
REGISTRATION_GEOMETRY_RESCUE_COVERAGE_GAIN = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_RESCUE_COVERAGE_GAIN", "0.04"))

OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))
DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))
GAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "5.4"))
GAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.052"))#.52
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2200"))
GAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))
GAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.1"))

OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "0") != "1"
OUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "7"))
OUTPUT_ADAPTIVE_MIN_TRACK_LEN = os.environ.get("BIOHUB_OUTPUT_ADAPTIVE_MIN_TRACK_LEN", "1") != "0"
OUTPUT_LOW_DENSITY_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_LOW_DENSITY_MIN_TRACK_LEN", "9"))
OUTPUT_DENSE_NODES_PER_FRAME = float(os.environ.get("BIOHUB_OUTPUT_DENSE_NODES_PER_FRAME", "500.0"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"

OUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.78"))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "3"))
GAP_REFINE_BG_PERCENTILE = float(os.environ.get("BIOHUB_GAP_REFINE_BG_PERCENTILE", "55.0"))
GAP_REFINE_CENTER_SIGMA_UM = float(os.environ.get("BIOHUB_GAP_REFINE_CENTER_SIGMA_UM", "1.60"))

OUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "1"
GAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "9.2"))
GAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "3.9"))
GAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0032"))
GAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "140"))
GAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.0045"))

OUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "4.7"))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "6.85"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "7.45"))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.0072"))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.00375"))


CONFIG_DISPLAY = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": METHOD,
    "weights": WEIGHTS_RELATIVE,
    "true_ensemble_models": [{k: v for k, v in m.items() if k != "path"} | {"path": m["path"]} for m in ENSEMBLE_MODELS],
    "target_artifact_slug": TARGET_ARTIFACT_SLUG,
    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),
    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,
    "det_threshold": DET_THRESHOLD,
    "edge_threshold": EDGE_THRESHOLD,
    "unet_batch_size": UNET_BATCH_SIZE,
    "use_ilp": USE_ILP,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "slice": SLICE,
    "allow_pip_install": ALLOW_PIP_INSTALL,
    "run_visual_eda": RUN_VISUAL_EDA,
    "run_dense_short7_diagnostics": RUN_DENSE_SHORT7_DIAGNOSTICS,
    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,
    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,
    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,
    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,
    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,
    "output_motion_relink": OUTPUT_MOTION_RELINK,
    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,
    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,
    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,
    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,
    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,
    "output_registration_aware": OUTPUT_REGISTRATION_AWARE,
    "registration_downsample_z": REGISTRATION_DOWNSAMPLE_Z,
    "registration_downsample_yx": REGISTRATION_DOWNSAMPLE_YX,
    "registration_min_psr": REGISTRATION_MIN_PSR,
    "registration_min_apply_um": REGISTRATION_MIN_APPLY_UM,
    "registration_max_apply_um": REGISTRATION_MAX_APPLY_UM,
    "registration_linefit_stabilize": REGISTRATION_LINEFIT_STABILIZE,
    "registration_geometry_validate": REGISTRATION_GEOMETRY_VALIDATE,
    "registration_geometry_min_nodes": REGISTRATION_GEOMETRY_MIN_NODES,
    "registration_geometry_distance_cap_um": REGISTRATION_GEOMETRY_DISTANCE_CAP_UM,
    "registration_geometry_coverage_um": REGISTRATION_GEOMETRY_COVERAGE_UM,
    "registration_geometry_veto_margin_um": REGISTRATION_GEOMETRY_VETO_MARGIN_UM,
    "registration_geometry_veto_coverage_drop": REGISTRATION_GEOMETRY_VETO_COVERAGE_DROP,
    "registration_geometry_rescue_min_um": REGISTRATION_GEOMETRY_RESCUE_MIN_UM,
    "registration_geometry_rescue_margin_um": REGISTRATION_GEOMETRY_RESCUE_MARGIN_UM,
    "registration_geometry_rescue_coverage_gain": REGISTRATION_GEOMETRY_RESCUE_COVERAGE_GAIN,
    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,
    "div_parent_max_um": DIV_PARENT_MAX_UM,
    "div_sister_max_um": DIV_SISTER_MAX_UM,
    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,
    "output_gap_close": OUTPUT_GAP_CLOSE,
    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,
    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),
    "gap_close_um": GAP_CLOSE_UM,
    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,
    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,
    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,
    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,
    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,
    "gap_refine_win_z": GAP_REFINE_WIN_Z,
    "gap_refine_win_yx": GAP_REFINE_WIN_YX,
    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,
    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,
    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,
    "output_adaptive_min_track_len": OUTPUT_ADAPTIVE_MIN_TRACK_LEN,
    "output_low_density_min_track_len": OUTPUT_LOW_DENSITY_MIN_TRACK_LEN,
    "output_dense_nodes_per_frame": OUTPUT_DENSE_NODES_PER_FRAME,
    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,
    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,
    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,
    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,
    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,
    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,
    "gap2_max_step_um": GAP2_MAX_STEP_UM,
    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,
    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,
    "gap2_require_context": GAP2_REQUIRE_CONTEXT,
    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,
    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,
    "safe_div_max_um": SAFE_DIV_MAX_UM,
    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,
    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,
    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,
    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,
}


print("Biohub A+B raw proposal exporter")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TRAIN_DIR:", TRAIN_DIR, "exists:", TRAIN_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))

In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]

# The safe path for offline reruns is to use attached wheels.
# Set BIOHUB_ALLOW_PIP_INSTALL=1 only for an interactive internet-enabled run.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)


In [ ]:
# Export exact shared A+B detections on all annotated training videos.
import numpy as np

EXPORT_DIR = WORKING_DIR / "biohub_ab_proposals"
EXPORT_ZIP = WORKING_DIR / "biohub_ab_proposals.zip"
EXPORT_MODELS_JSON = WORKING_DIR / "ab_export_models.json"
EXPORT_SCRIPT_PATH = REPO_DIR / "scripts" / "export_ab_proposals.py"

selected_models = [
    {
        "name": m["name"], "enabled": bool(m["enabled"]),
        "path": m["path"], "weight": float(m["weight"]), "device": m["device"],
    }
    for m in ENSEMBLE_MODELS[:2]
]
if len(selected_models) != 2 or not all(m["enabled"] for m in selected_models):
    raise RuntimeError("Proposal export requires A and B enabled")
for member in selected_models:
    if not Path(member["path"]).exists():
        raise FileNotFoundError(f"{member['name']} checkpoint missing: {member['path']}")

train_stems = sorted(p.name[:-5] for p in TRAIN_DIR.glob("*.zarr"))
if len(train_stems) != 199:
    raise RuntimeError(f"Expected 199 train videos, found {len(train_stems)} under {TRAIN_DIR}")
print("Training videos:", len(train_stems))
print("Detector members:", selected_models)

EXPORT_MODELS_JSON.write_text(json.dumps(selected_models, indent=2))
EXPORT_SCRIPT_PATH.write_text('#!/usr/bin/env python3\n"""Export raw shared A+B detections for proposal-aware association training.\n\nNo edge prediction, ILP, or graph post-processing is run. Each video becomes a\ncompressed NPZ containing raw-voxel coordinates, fused/member detection\nconfidence, frame offsets, and downsample-identical transitions.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport gc\nimport hashlib\nimport json\nimport shutil\nimport sys\nimport time\nimport zipfile\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport zarr\nfrom tqdm import tqdm\n\nsys.path.insert(0, str(Path(__file__).parent))\nimport predict_unet_transformer as base\nfrom biohub_tracking.io import open_dataset\n\n\n@dataclass\nclass Member:\n    name: str\n    path: Path\n    weight: float\n    device: torch.device\n    model: object\n    window_size: int\n    downsample: tuple[int, ...]\n\n\ndef parse_args():\n    p = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)\n    p.add_argument("--data-dir", type=Path, required=True)\n    p.add_argument("--models-json", type=Path, required=True)\n    p.add_argument("--output-dir", type=Path, default=Path("/kaggle/working/biohub_ab_proposals"))\n    p.add_argument("--det-threshold", type=float, default=0.99)\n    p.add_argument("--pool-kernel-um", type=float, default=5.0)\n    p.add_argument("--slice", default="", help="Python slice, for example :10 or 50:100")\n    p.add_argument("--resume", action="store_true")\n    p.add_argument("--zip", action="store_true")\n    return p.parse_args()\n\n\ndef sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef load_members(spec_path: Path) -> list[Member]:\n    specs = json.loads(spec_path.read_text())\n    members = []\n    for spec in specs:\n        if not spec.get("enabled", True) or float(spec.get("weight", 0)) <= 0:\n            continue\n        path = Path(spec["path"])\n        if not path.exists():\n            raise FileNotFoundError(f"{spec[\'name\']} checkpoint missing: {path}")\n        requested = str(spec.get("device", "cuda:0"))\n        device = torch.device(requested if torch.cuda.is_available() else "cpu")\n        model, window, downsample = base.load_model(path, device)\n        members.append(Member(str(spec["name"]), path, float(spec["weight"]),\n                              device, model, int(window), tuple(downsample)))\n        print(f"Loaded {spec[\'name\']} weight={spec[\'weight\']} device={device}: {path}", flush=True)\n    if not members:\n        raise ValueError("No enabled detector members")\n    reference = (members[0].window_size, members[0].downsample)\n    for m in members[1:]:\n        if (m.window_size, m.downsample) != reference:\n            raise ValueError(f"Detector configs differ: {members[0].name}={reference}, "\n                             f"{m.name}={(m.window_size, m.downsample)}")\n    return members\n\n\ndef weighted_average(values, weights, device):\n    total = float(sum(weights))\n    out = values[0].to(device, non_blocking=True) * (weights[0] / total)\n    for value, weight in zip(values[1:], weights[1:]):\n        out = out + value.to(device, non_blocking=True) * (weight / total)\n    return out\n\n\ndef sample_probs(logits: torch.Tensor, coords: np.ndarray) -> np.ndarray:\n    if len(coords) == 0:\n        return np.empty(0, dtype=np.float32)\n    # encode() returns a batch dimension and a singleton detection channel;\n    # reduce to the spatial Z,Y,X logit volume before advanced indexing.\n    volume = logits\n    while volume.ndim > 3:\n        volume = volume[0]\n    if volume.ndim != 3:\n        raise ValueError(f"Unexpected detection-logit shape: {tuple(logits.shape)}")\n    index = torch.as_tensor(coords[:, 1:].astype(np.int64), device=logits.device)\n    probs = torch.sigmoid(volume[index[:, 0], index[:, 1], index[:, 2]])\n    return probs.detach().float().cpu().numpy()\n\n\n@torch.no_grad()\ndef export_video(members: list[Member], path: Path, output_path: Path,\n                 det_threshold: float, pool_kernel_um: float) -> dict:\n    started = time.time()\n    primary = members[0].device\n    weights = [m.weight for m in members]\n    W = members[0].window_size\n    downsample = members[0].downsample\n\n    ds = open_dataset(path, normalize=False, load_image=False, downsample=downsample)\n    if "0.001" not in ds.quantiles or "0.999" not in ds.quantiles:\n        raise ValueError(f"Missing 0.001/0.999 quantiles: {path}")\n    zarr_arr = zarr.open_group(str(ds.zarr_path), mode="r")["0"]\n    T = int(ds.image_shape[0])\n    target_shape = list(ds.image_shape[1:])\n    q_low, q_high = float(ds.quantiles["0.001"]), float(ds.quantiles["0.999"])\n    voxel_size = tuple(s * d for s, d in zip(ds.scale, downsample))\n    pool_kernel = base.pool_kernel_from_um(pool_kernel_um, voxel_size)\n\n    # One cached strided read per frame. This both avoids duplicate I/O across\n    # overlapping model windows and identifies frozen transitions.\n    frames = [base._load_frame(zarr_arr, t, target_shape, downsample) for t in range(T)]\n    frozen_sources = np.asarray(\n        [t for t in range(T - 1) if torch.equal(frames[t], frames[t + 1])],\n        dtype=np.int16,\n    )\n\n    stride = max(W - 1, 1)\n    starts = list(range(0, T - W + 1, stride))\n    if not starts or starts[-1] + W < T:\n        last = max(T - W, 0)\n        if not starts or last != starts[-1]:\n            starts.append(last)\n\n    seen = set()\n    coords_by_t: dict[int, np.ndarray] = {}\n    fused_prob_by_t: dict[int, np.ndarray] = {}\n    member_prob_by_t: dict[int, np.ndarray] = {}\n    for ws in starts:\n        frame_indices = list(range(ws, ws + W))\n        imgs_cpu = torch.stack([frames[t] for t in frame_indices])\n        imgs_cpu = ((imgs_cpu - q_low) / (q_high - q_low + 1e-6)).clamp(0.0).unsqueeze(0)\n        member_dets = []\n        for member in members:\n            imgs = imgs_cpu.to(member.device, non_blocking=True)\n            _features, det = member.model.encode(imgs)\n            member_dets.append(det)\n        for fi, t in enumerate(frame_indices):\n            if t in seen:\n                continue\n            fused_logits = weighted_average(\n                [member_dets[k][fi] for k in range(len(members))], weights, primary,\n            )\n            coords = base._detect_cells_pooled(\n                fused_logits[0], t, det_threshold, pool_kernel,\n            )\n            coords_by_t[t] = coords\n            fused_prob_by_t[t] = sample_probs(fused_logits, coords)\n            member_prob_by_t[t] = np.stack([\n                sample_probs(member_dets[k][fi], coords)\n                for k in range(len(members))\n            ], axis=1) if len(coords) else np.empty((0, len(members)), np.float32)\n            seen.add(t)\n        del member_dets, imgs_cpu\n\n    if len(seen) != T:\n        raise RuntimeError(f"{path.stem}: detected {len(seen)}/{T} frames")\n    counts = np.asarray([len(coords_by_t[t]) for t in range(T)], dtype=np.int32)\n    offsets = np.concatenate([[0], np.cumsum(counts, dtype=np.int64)])\n    coords = np.concatenate([coords_by_t[t] for t in range(T)]).astype(np.int16)\n    coords[:, 1:] *= np.asarray(downsample, dtype=np.int16)\n    fused_prob = np.concatenate([fused_prob_by_t[t] for t in range(T)]).astype(np.float32)\n    member_prob = np.concatenate([member_prob_by_t[t] for t in range(T)]).astype(np.float32)\n    if not (len(coords) == len(fused_prob) == len(member_prob)):\n        raise AssertionError("Proposal arrays have inconsistent lengths")\n\n    np.savez_compressed(\n        output_path,\n        coords=coords,\n        fused_det_prob=fused_prob,\n        member_det_prob=member_prob,\n        frame_counts=counts,\n        frame_offsets=offsets,\n        frozen_sources=frozen_sources,\n        image_shape=np.asarray((T, *zarr_arr.shape[1:]), dtype=np.int32),\n        voxel_scale_um=np.asarray(ds.scale, dtype=np.float32),\n        downsample=np.asarray(downsample, dtype=np.int16),\n        det_threshold=np.asarray(det_threshold, dtype=np.float32),\n        pool_kernel_um=np.asarray(pool_kernel_um, dtype=np.float32),\n    )\n    del frames, coords_by_t, fused_prob_by_t, member_prob_by_t\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return {\n        "dataset": path.stem,\n        "frames": T,\n        "nodes": len(coords),\n        "mean_nodes_per_frame": float(counts.mean()),\n        "min_nodes_per_frame": int(counts.min()),\n        "max_nodes_per_frame": int(counts.max()),\n        "frozen_transitions": len(frozen_sources),\n        "seconds": time.time() - started,\n        "file": output_path.name,\n        "bytes": output_path.stat().st_size,\n    }\n\n\ndef main():\n    args = parse_args()\n    torch.set_float32_matmul_precision("high")\n    torch.backends.cuda.matmul.allow_tf32 = True\n    members = load_members(args.models_json)\n    stems = sorted(p.name[:-5] for p in args.data_dir.glob("*.zarr"))\n    if args.slice:\n        parts = [int(x) if x else None for x in args.slice.split(":")]\n        stems = stems[slice(*parts)]\n    if not stems:\n        raise FileNotFoundError(f"No .zarr videos selected under {args.data_dir}")\n    if args.output_dir.exists() and not args.resume:\n        shutil.rmtree(args.output_dir)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    rows = []\n    manifest_path = args.output_dir / "manifest.csv"\n    for i, stem in enumerate(stems, 1):\n        out = args.output_dir / f"{stem}.npz"\n        if args.resume and out.exists():\n            print(f"[{i:3d}/{len(stems)}] SKIP {stem}", flush=True)\n            continue\n        row = export_video(\n            members, args.data_dir / stem, out,\n            args.det_threshold, args.pool_kernel_um,\n        )\n        rows.append(row)\n        print(f"[{i:3d}/{len(stems)}] {stem}: nodes={row[\'nodes\']:,} "\n              f"frozen={row[\'frozen_transitions\']} time={row[\'seconds\']/60:.1f}m", flush=True)\n        # Incremental manifest survives a late-session interruption.\n        existing = []\n        if manifest_path.exists():\n            with manifest_path.open(newline="") as f:\n                existing = list(csv.DictReader(f))\n        by_name = {str(r["dataset"]): r for r in existing}\n        by_name[stem] = row\n        fields = list(row)\n        with manifest_path.open("w", newline="") as f:\n            writer = csv.DictWriter(f, fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(by_name[k] for k in sorted(by_name))\n\n    model_info = [{\n        "name": m.name, "path": str(m.path), "weight": m.weight,\n        "device": str(m.device), "sha256": sha256(m.path),\n        "window_size": m.window_size, "downsample": list(m.downsample),\n    } for m in members]\n    manifest = {\n        "format": "biohub_ab_proposals_v1",\n        "data_dir": str(args.data_dir),\n        "selected_videos": len(stems),\n        "det_threshold": args.det_threshold,\n        "pool_kernel_um": args.pool_kernel_um,\n        "models": model_info,\n        "notes": "Raw A+B shared detections before edge prediction, ILP, or post-processing.",\n    }\n    (args.output_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))\n\n    if args.zip:\n        zip_path = args.output_dir.parent / "biohub_ab_proposals.zip"\n        print(f"Creating {zip_path} ...", flush=True)\n        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED,\n                             compresslevel=3, allowZip64=True) as zf:\n            for path in sorted(args.output_dir.iterdir()):\n                zf.write(path, arcname=f"biohub_ab_proposals/{path.name}")\n        print(f"Archive: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)", flush=True)\n    print(f"Done: {len(stems)} videos -> {args.output_dir}")\n\n\nif __name__ == "__main__":\n    main()\n')

cmd = [
    sys.executable, str(EXPORT_SCRIPT_PATH),
    "--data-dir", str(TRAIN_DIR),
    "--models-json", str(EXPORT_MODELS_JSON),
    "--output-dir", str(EXPORT_DIR),
    "--det-threshold", str(DET_THRESHOLD),
    "--pool-kernel-um", "5.0",
    "--zip",
]
if SLICE:
    cmd += ["--slice", SLICE]

started = time.time()
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": "src"}, check=True)
elapsed = time.time() - started

manifest_csv = pd.read_csv(EXPORT_DIR / "manifest.csv")
manifest_json = json.loads((EXPORT_DIR / "manifest.json").read_text())
print(manifest_csv.to_string(index=False))
print(json.dumps(manifest_json, indent=2))
print(f"Export runtime: {elapsed/3600:.2f} hours")
print(f"Total proposals: {int(manifest_csv['nodes'].sum()):,}")
print(f"Frozen transitions: {int(manifest_csv['frozen_transitions'].sum()):,}")

# Integrity-check every archive before discarding the duplicate loose directory.
for row in manifest_csv.itertuples(index=False):
    path = EXPORT_DIR / row.file
    with np.load(path) as item:
        n = len(item["coords"])
        assert n == len(item["fused_det_prob"]) == len(item["member_det_prob"])
        assert int(item["frame_counts"].sum()) == n
        assert int(item["frame_offsets"][-1]) == n
        assert item["member_det_prob"].shape[1] == 2

shutil.copy2(EXPORT_DIR / "manifest.csv", WORKING_DIR / "ab_proposals_manifest.csv")
shutil.copy2(EXPORT_DIR / "manifest.json", WORKING_DIR / "ab_proposals_manifest.json")
(WORKING_DIR / "ab_proposals_export_summary.json").write_text(json.dumps({
    "videos": len(manifest_csv),
    "nodes": int(manifest_csv["nodes"].sum()),
    "frozen_transitions": int(manifest_csv["frozen_transitions"].sum()),
    "runtime_seconds": elapsed,
    "archive": str(EXPORT_ZIP),
    "archive_bytes": EXPORT_ZIP.stat().st_size,
}, indent=2))

# results.zip only needs the inner archive plus small manifests, not a duplicate
# copy of all 199 loose NPZ files.
shutil.rmtree(EXPORT_DIR)
print("READY:", EXPORT_ZIP, f"({EXPORT_ZIP.stat().st_size/1e6:.1f} MB)")
